In [3]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [4]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [480]:
def get_text(driver, selector):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return el.text.strip()
    except:
        return None

In [5]:
def get_section_detail(driver, title):
    try:
        
        # Tìm container chứa nội dung
        container = driver.find_element(By.CSS_SELECTOR, "div.render-html.text-14")
        
        # Lấy tất cả h3 trong container
        h3s = container.find_elements(By.TAG_NAME, "h3")
        title_norm = title.lower().strip()
        
        for h3 in h3s:
            h3_text = h3.get_attribute("textContent") or ""
            
            if title_norm in h3_text.lower():
                # Lấy siblings trực tiếp từ h3
                siblings = h3.find_elements(By.XPATH, "./following-sibling::*")
                
                content = []
                for sib in siblings:
                    tag = sib.tag_name.lower()
                    
                    # Dừng khi gặp h3 tiếp theo
                    if tag == "h3":
                        break
                    
                    # Lấy text
                    t = sib.get_attribute("textContent") or ""
                    t = t.strip()
                    if t:
                        content.append(t)
                
                return "\n".join(content) if content else None
                
    except:
        pass
    return None

In [6]:
def get_product_images(driver):
    images = []

    try:
        img_elements = driver.find_elements(By.CSS_SELECTOR, "img[src*='Products/Images'], img[src*='cdn.tgdd.vn']")

        for img in img_elements:
            src = img.get_attribute("src")
            if src and "Products/Images"in src:
                if "https://cdn.tgdd.vn" in src:
                    match = re.search(r'(https://cdn\.tgdd\.vn/Products/Images/[^\s]+)', src)
                    if match:
                        original_url = match.group(1)
                        if original_url not in images:
                            images.append(original_url)
                elif src not in images:
                    images.append(src)
    except Exception as e:
        print(f"Error {e}")
        pass
    return images

In [7]:
# Lấy thông tin từ bảng grid
def get_product_infomation(driver, label):
    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "#contentRef .grid.grid-cols-3")
        for row in rows:
            label_el = row.find_element(By.CSS_SELECTOR, "div.text-13.font-bold")
            label_text = label_el.get_attribute("textContent") or ""
            
            if label.lower() in label_text.lower():
                value_el = row.find_element(By.CSS_SELECTOR, "div.col-span-2 div.mr-2")
                value_text = value_el.get_attribute("textContent") or ""
                return value_text.strip()
    except Exception as e:
        print(f"Get product information error: {e}")
        pass
    return None

In [8]:
def get_medicine_data(driver):
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[class*='text-20'][class*='font-bold']")))
    except:
        pass

    name = get_text(driver, "div.text-20.font-bold")
    if not name:
        try:
            name = driver.title.split(' - ')[0].strip()
        except:
            name = None

    price = None
    try:
        price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")
        price = price_el.text.strip()
    except:
        pass
    

    images = get_product_images(driver)


    expand_button_1 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[6]/button")
    expand_button_1.click()
    time.sleep(2)

    cong_dung_tom_tat = get_product_infomation(driver, "Công dụng")
    thanh_phan_chinh = get_product_infomation(driver, "Thành phần chính")
    doi_tuong = get_product_infomation(driver, "Đối tượng sử dụng")
    thuong_hieu = get_product_infomation(driver, "Thương hiệu")
    noi_san_xuat = get_product_infomation(driver, "Nơi sản xuất")
    dang_bao_che = get_product_infomation(driver, "Dạng bào chế")
    han_dung = get_product_infomation(driver, "Hạn dùng")


    close_btn = driver.find_element(By.CSS_SELECTOR, "button[class*='absolute'][class*='right']")
    close_btn.click()
    time.sleep(2)



    expand_button_2 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[4]/button")
    expand_button_2.click()
    time.sleep(2)

    thanh_phan_chi_tiet = get_section_detail(driver, "1. Thành phần")
    cong_dung_chi_tiet = get_section_detail(driver, "2. Công dụng")
    cach_dung = get_section_detail(driver, "3. Cách dùng")
    chong_chi_dinh = get_section_detail(driver, "4. Chống chỉ định")
    tac_dung_phu = get_section_detail(driver, "5. Tác dụng phụ")
    luu_y = get_section_detail(driver, "6. Lưu ý")

    return {
        "ten_thuoc": name,
        "gia": price,
        "hinh_anh": "|".join(images) if images else None,
        "cong_dung_tom_tat": cong_dung_tom_tat,
        "thanh_phan_chinh": thanh_phan_chinh,
        "doi_tuong": doi_tuong,
        "thuong_hieu": thuong_hieu,
        "noi_san_xuat": noi_san_xuat,
        "dang_bao_che": dang_bao_che,
        "han_dung": han_dung,
        "thanh_phan_chi_tiet": thanh_phan_chi_tiet,
        "cong_dung_chi_tiet": cong_dung_chi_tiet,
        "cach_dung": cach_dung,
        "chong_chi_dinh": chong_chi_dinh,
        "tac_dung_phu": tac_dung_phu,
        "luu_y": luu_y
    }


<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_3200\1769263606.py:16: SyntaxWarning: invalid escape sequence '\['
  price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")


In [9]:
def crawl_ankhang_medicines(csv_path, out_csv):
    df = pd.read_csv(csv_path)
    driver = set_up_driver()
    results = []

    try:

        for i, row in df.iterrows():
            url = row['link']
            category = row.get('category', '')
            drug_type = row.get('drug_type', '')

            try:
                driver.get(url)
                data = get_medicine_data(driver)
                data['url'] = url
                data['category'] = category
                data['drug_type'] = drug_type
                results.append(data)
                print(f"[{i+1}/{len(df)}] crawled: {data['ten_thuoc']}")
            except Exception as e:
                results.append({'url': url, 'error': str(e)})
                print(f"[{i+1}/{len(df)}] failed: {url} with error {e}")

            time.sleep(2)  

            if (i + 1) % 50 == 0:
                pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
                print(f"Saved {len(results)} records to {out_csv}")

        
    finally:
        driver.quit()
        pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
        print(f"Đã lưu {len(results)} sản phẩm vào {out_csv}")

 

In [486]:
driver = set_up_driver()
driver.get("https://www.nhathuocankhang.com/thuoc-bo-va-vitamin/thuoc-bo-sung-vitamin-b-timi-roitin-120-vien?sku=1193668000314")
time.sleep(3)
data = get_medicine_data(driver)
print(data)
driver.quit()

{'ten_thuoc': 'Timi Roitin trị viêm dây thần kinh, thoái hóa khớp (24 vỉ x 5 viên)', 'gia': '528.000₫', 'hinh_anh': 'https://cdn.tgdd.vn/Products/Images/10053/130480/timi-roitin-2-1.jpg|https://cdn.tgdd.vn/Products/Images/10053/130480/timi-roitin-3-1.jpg|https://cdn.tgdd.vn/Products/Images/10053/130480/timi-roitin-0-1.jpg', 'cong_dung_tom_tat': 'Bổ sung vitamin nhóm B, điều trị viêm dây thần kinh ngoại biên, viêm đa dây thần kinh,...', 'thanh_phan_chinh': 'Chondroitin, Vitamin B5, Fursultiamine, Vitamin PP, Vitamin B6, Vitamin B2', 'doi_tuong': 'Người lớn', 'thuong_hieu': 'Phil Inter Pharma', 'noi_san_xuat': 'Việt Nam', 'dang_bao_che': 'Viên nang mềm', 'han_dung': '36 tháng kể từ ngày sản xuất', 'thanh_phan_chi_tiet': 'Hoạt chất: Chondroitin sulfate natri 90mg, Nicotinamide 50mg, Fursultiamine 50mg, Riboflavin 6mg, Pyridoxine HCl 25mg, Calci pantothenate 15mg.\nTá dược: Dầu đậu nành, Sáp ong trắng, Dầu Lecithin, Dầu cọ, Gelatin, Glycerin đậm đặc, D-sorbitol 70%, Ethyl vanillin, Màu vàn

In [487]:
crawl_ankhang_medicines("ankhang_medicine_links.csv", "ankhang_medicines_data_demo.csv")

[1/2127] crawled: Timi Roitin trị viêm dây thần kinh, thoái hóa khớp (24 vỉ x 5 viên)
[2/2127] crawled: B Complex C bổ sung vitamin nhóm B và vitamin C (10 vỉ x 10 viên)
[3/2127] failed: https://www.nhathuocankhang.com/thuoc-bo-va-vitamin/cezinco-110mg-5ml-h-30-ong?sku=1193668000849 with error Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id='contentRef']/div/div[4]/button"}
  (Session info: chrome=145.0.7632.45); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff721e94325
	0x7ff721e94380
	0x7ff721c2d49d
	0x7ff721c882fe
	0x7ff721c8860c
	0x7ff721cd8b67
	0x7ff721cd5728
	0x7ff721c78e38
	0x7ff721c79d23
	0x7ff722163090
	0x7ff72215d918
	0x7ff72217dc8a
	0x7ff721eb06d5
	0x7ff721eb967c
	0x7ff721e9d684
	0x7ff721e9d836
	0x7ff721e83347
	0x7ffd717ae8d7
	0x7ffd722cc40c

[4/2127] crawled: Vitamin PP 

In [10]:
ankhang_medicine_data = pd.read_csv("ankhang_medicines_data_demo.csv")
ankhang_medicine_data.head(10)

,ten_thuoc,gia,hinh_anh,cong_dung_tom_tat,thanh_phan_chinh,doi_tuong,thuong_hieu,noi_san_xuat,dang_bao_che,han_dung,thanh_phan_chi_tiet,cong_dung_chi_tiet,cach_dung,chong_chi_dinh,tac_dung_phu,luu_y,url,category,drug_type,error
0,"Timi Roitin trị viêm dây thần kinh, thoái hóa ...",528.000₫,https://cdn.tgdd.vn/Products/Images/10053/1304...,"Bổ sung vitamin nhóm B, điều trị viêm dây thần...","Chondroitin, Vitamin B5, Fursultiamine, Vitami...",Người lớn,Phil Inter Pharma,Việt Nam,Viên nang mềm,36 tháng kể từ ngày sản xuất,"Hoạt chất: Chondroitin sulfate natri 90mg, Nic...",- Bổ sung các vitamin nhóm B trong các trường ...,Người lớn: Uống 1 viên/ngày.\n- Quá liều\nDùng...,- Quá mẫn với bất cứ thành phần nào của thuốc....,TIMIROITIN thường được dung nạp tốt khi dùng ở...,- Thận trọng khi sử dụng\n- Khi sử dụng nicoti...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
1,B Complex C bổ sung vitamin nhóm B và vitamin ...,800₫,https://cdn.tgdd.vn/Products/Images/10053/1295...,Dự phòng & bổ sung khi thiếu hụt các vitamin n...,"Vitamin PP, Vitamin C, Vitamin B6, Vitamin B2,...",NaN,Vidipha,Việt Nam,Viên nang cứng,24 tháng kể từ ngày sản xuất,"Hoạt chất: Vitamin B1 15mg, Vitamin B2 10mg, V...",B Complex C dự phòng và bổ sung thiếu hụt các ...,Liều lượng: Trung bình: 1 - 2 viên/ngày.\nCách...,Dị ứng với một trong các thành phần của thuốc....,Dùng liều cao nước tiểu sẽ có màu vàng nhạt (d...,- Thận trọng khi sử dụng\nKhi sử dụng nicotina...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
3,Vitamin PP 50 Pharmedic ngăn ngừa thiếu nicoti...,12.000₫,https://cdn.tgdd.vn/Products/Images/10053/1533...,Bổ sung vào khẩu phần ăn để ngăn ngừa thiếu hụ...,Vitamin PP,Người lớn và trẻ em trên 5 tuổi,Pharmedic,Việt Nam,Viên nén,NaN,Nicotinamid 50mg.\nTá dược 1 viên.,Vitamin PP 50 giúp bổ sung vào khẩu phần ăn để...,"Người lớn: mỗi lần 1 - 2 viên, ngày 3 lần.\nTr...","- Dị ứng với nicotinamid.\n- Bệnh gan nặng, lo...",NaN,- Thận trọng khi sử dụng\nNgười có tiền sử loé...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
4,"Agi-Calci bổ sung canxi, trị loãng xương (20 v...",150.000₫,https://cdn.tgdd.vn/Products/Images/10053/2463...,Bổ sung calci khi thiếu hay tăng nhu cầu calci...,"Calci cacbonat, Vitamin D3",NaN,Agimexpharm,Việt Nam,Viên nén bao phim,24 tháng kể từ ngày sản xuất,Mỗi viên nén bao phim chứa:\nHoạt chất: Calci ...,- Agi-Calci bổ sung calci trong các trường hợp...,Uống thuốc buổi sáng hoặc buổi trưa theo liều ...,"- Tăng calci huyết, calci niệu, sỏi calci, suy...",- Dùng thuốc chứa muối calci qua đường uống có...,- Thận trọng khi sử dụng\nThận trọng khi sử dụ...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
5,Vitamin B1 Mekophar 250mg trị các bệnh thiếu v...,7.000₫,https://cdn.tgdd.vn/Products/Images/10053/1535...,"Điều trị các bệnh do thiếu B1, suy nhược cơ th...",NaN,NaN,Mekophar,Việt Nam,Viên nén bao đường,36 tháng kể từ ngày sản xuất,Hoạt chất: Thiamine nitrate (vitamin B1) 250mg...,- Điều trị các bệnh do thiếu vitamin B1 như: b...,- Theo chỉ dẫn của bác sĩ.\n- Người lớn: uống ...,Quá mẫn với một trong các thành phần của thuốc.,"- Tác dụng phụ rất hiếm khi xảy ra như: ngứa, ...",- Thận trọng khi sử dụng\nThận trọng khi sử dụ...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
6,Dung dịch uống Calciumboston Ascorbic trị thiế...,9.100₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị các tình trạng thiếu calci, vitamin C...","Vitamin PP, Vitamin C, Calci glucoheptonat",Người lớn và trẻ em,Boston Pharma,Việt Nam,Dung dịch,36 tháng kể từ ngày sản xuất,Hoạt chất: Mỗi ml dung dịch thuốc chứa:\n- Cal...,Điều trị các tình trạng thiếu vitamin C và Vit...,- Cách dùng\nCalciu

In [12]:
# Lấy nhưng dòng bị lỗi crawl lại
error_rows = ankhang_medicine_data[ankhang_medicine_data['ten_thuoc'].isnull() | ankhang_medicine_data['ten_thuoc'].isna()]
print(f"Số dòng bị lỗi: {error_rows.shape[0]}")
error_rows.head()

Số dòng bị lỗi: 54


,ten_thuoc,gia,hinh_anh,cong_dung_tom_tat,thanh_phan_chinh,doi_tuong,thuong_hieu,noi_san_xuat,dang_bao_che,han_dung,thanh_phan_chi_tiet,cong_dung_chi_tiet,cach_dung,chong_chi_dinh,tac_dung_phu,luu_y,url,category,drug_type,error
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-tri-ung-...,NaN,NaN,Message: no such element: Unable to locate ele...
106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...


In [13]:
error_rows.to_csv("ankhang_medicines_error_rows.csv", index=False, encoding='utf-8-sig')